# Module 02 — Plan-and-Execute Agent Executor

**Released:** v1.14.0 (April 7, 2026)

The new executor flips a single ReAct loop into three cooperating pieces:

- **Planner** — decomposes the task into an ordered `TodoList` of `PlanStep`s.
- **StepExecutor** — runs each step in its own bounded multi-turn loop (default 15 iterations). It does **not** read outer state.
- **PlannerObserver** — evaluates each step's result and decides: *continue*, *replan now*, or *done*.

You enable it by passing a `PlanningConfig` to the agent.

**This module's demo** wires the Plan-and-Execute executor to Daytona sandbox tools (`DaytonaFileTool`, `DaytonaExecTool`) and asks the agent to **build a working todo-list web app** — planner decomposes it into file-write steps, executor writes each file into the sandbox, and at the end we render the live preview.

In [ ]:
from dotenv import load_dotenv
load_dotenv()

from crewai.agent.planning_config import PlanningConfig

PlanningConfig.model_fields.keys()

## PlanningConfig — the knobs that matter

| Field | Default | Effect |
|---|---|---|
| `reasoning_effort` | `"medium"` | `low` = observer only replans on hard failures. `medium` = replans on any step failure. `high` = replans aggressively + early-goal detection. |
| `max_attempts` | `None` | Cap refinement cycles. `None` keeps going until ready. |
| `max_steps` | `20` | Max steps the planner can emit. |
| `llm` | agent's LLM | Let the planner use a cheaper/smarter model than the executor. |

Tune `reasoning_effort` first — it moves the speed/quality trade-off the most.

In [ ]:
config = PlanningConfig(reasoning_effort="medium", max_attempts=3, max_steps=8)
config

## Run the Flow

`AppBuilderFlow` provisions a fresh Daytona sandbox, lets the planner break "build a todo-list web app" into concrete file-write steps, executes each via the Daytona tools, then exposes a preview URL on port 8000.

Watch the logs for `plan generated → step 1 (write index.html) → observe → step 2 (write style.css) → ...`.

**Prereqs:** `DAYTONA_API_KEY` in your env (grab one at [daytona.io](https://daytona.io)).

In [ ]:
from showcase.flows.planning_flow import AppBuilderFlow

flow = AppBuilderFlow()
flow.kickoff()
print("sandbox_id: ", flow.state.sandbox_id)
print("preview_url:", flow.state.preview_url)

## See the app

Two ways to look at what the agent built:

1. **Live preview** — embed the Daytona preview URL in an iframe. Requires the sandbox to still be running (it stays up until you delete it or the Daytona timeout elapses).
2. **Inline HTML** — the Flow snapshots `index.html` into `state.rendered_html` so you can render it right here without the sandbox.

In [ ]:
from IPython.display import IFrame, HTML, display

# Live preview from Daytona (requires the sandbox to still be up)
display(IFrame(flow.state.preview_url, width=720, height=520))

In [ ]:
# Inline fallback — renders the snapshotted index.html locally (style.css and
# app.js won't be loaded because they live inside the sandbox, but it lets you
# inspect what the agent actually wrote).
display(HTML(flow.state.rendered_html or "<em>no html snapshotted</em>"))